## Setup

In [1]:
from evosax import Strategies, ParameterReshaper, EvoState

In [2]:
from jaxcmr.memorysearch import FittingStrategy, BaseCMR, InstanceCMR, create_predict_fn, predict_trials, predict_trial, experience, start_retrieving
import pytest
import jax
import json
from jax import numpy as jnp, vmap, tree_util, random, lax, jit
from jax.nn import sigmoid
from jaxcmr.datasets import load_data, generate_trial_mask
from jaxcmr.helpers import latin_hypercube_sampling, scale_to_bounds, log_likelihood
from functools import partial

In [3]:
parameter_path = "D:/data/base_cmr_parameters.json"
data_path = "D:/data/{}.h5"

model_create_fn = BaseCMR
data_tag = "HealyKahana2014"
trial_query = "data['listtype'] == -1"
rng = jax.random.PRNGKey(0)

with open(parameter_path, "r") as f:
    parameters = json.load(f)

data = load_data(data_path.format(data_tag))
trial_mask = generate_trial_mask(data, trial_query)
subject_trials = jnp.array(data["recalls"][trial_mask].reshape(126, 28, 16))
subject_presentations = jnp.array(data["pres_itemnos"][trial_mask].reshape(126, 28, 16))

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


## Data Likelihood

In [4]:
item_count = 16

model = start_retrieving(experience(BaseCMR(item_count, item_count, parameters['fixed'])))
log_likelihood(vmap(predict_trial, in_axes=(None, 0))(model, subject_trials[0])[1])

Array(712.90137, dtype=float32)

## Base Example

In [5]:
# def scale_params(params: dict, bounds: dict[str, list[float]]):
#     return tree_util.tree_map(lambda x, y: y[0] + x * (y[1] - y[0]), params, bounds)


def scale_params(params: dict, bounds: dict[str, list[float]]):
    return jax.tree_util.tree_map(
        lambda x, bound: bound[0] + sigmoid(x) * (bound[1] - bound[0]), params, bounds
    )


reshape_params = ParameterReshaper(
    {key: parameters["fixed"][key] for key in parameters["free"]}
).reshape

ParameterReshaper: 12 parameters detected for optimization.


In [6]:
subject_index = 11
popsize = 50

assert subject_index > 0

for s_name in ["DE"]:
# for s_name in ["SimpleES", "SimpleGA", "PSO", "DE", "Sep_CMA_ES",
#             "Full_iAMaLGaM", "Indep_iAMaLGaM", "MA_ES", "LM_MA_ES",
#             "RmES", "GLD", "SimAnneal", "GESMR_GA", "SAMR_GA"]:
    strategy = Strategies[s_name](popsize=popsize, num_dims=len(parameters["free"]))
    es_params = strategy.default_params
    state = strategy.initialize(rng, es_params)

    single_subject_objective_fn = vmap(
        create_predict_fn(
            model_create_fn,
            subject_presentations[subject_index-1],
            subject_trials[subject_index-1],
        )
    )

    for t in range(1000):
        rng, rng_iter = random.split(rng)

        if t == 0:
            x = scale_to_bounds(
                latin_hypercube_sampling(rng_iter, len(parameters["free"]), popsize), -5, 5)
            state = state.replace(archive=x)
        else:
            x, state = strategy.ask(rng_iter, state, es_params)

        x_dict = scale_params(reshape_params(x), parameters["free"])
        fitness = single_subject_objective_fn(x_dict)
        state = strategy.tell(x, fitness, state, es_params)

        if (t + 1) % 50 == 0:
            print(
                "{} - # Gen: {}|Fitness: {:.2f}|Params: {}".format(
                    s_name, t + 1, state.best_fitness, state.best_member
                )
            )
    print(20 * "=")

DE - # Gen: 50|Fitness: 845.82|Params: [  5.2544312  10.143901   -5.3206186  -2.6816807   1.5096127   2.3627815
 -13.035623    1.48625     3.4274263   3.8181336  -3.8179433  -4.476961 ]
DE - # Gen: 100|Fitness: 832.16|Params: [  87.979576    31.276972    -7.233371    -7.3171787    4.2411466
   23.568344    -9.011524    -7.126436    12.240529  -107.697075
   -3.0584478   -8.1304865]
DE - # Gen: 150|Fitness: 830.89|Params: [ 577.7954      54.291245    -5.5620584  -11.178938     2.1165223
  161.86598     -8.881226    -9.219122    17.324564  -266.07483
   -3.0516262   -8.033874 ]
DE - # Gen: 200|Fitness: 830.89|Params: [ 475.7563      27.36668     -5.6127396  -15.819034     2.177738
  181.2432      -8.869331   -11.283995    25.414785  -370.46985
   -3.043474    -8.07639  ]
DE - # Gen: 250|Fitness: 830.89|Params: [ 520.12195     28.515915    -5.6121616  -18.752594     2.1779468
  266.92017     -8.869357   -11.688878    25.98826   -140.98193
   -3.0444953   -8.074737 ]
DE - # Gen: 300|Fitnes

## Compiled Scan

In [18]:
def latin_hypercube_initialize(state, rng):
    x = scale_to_bounds(latin_hypercube_sampling(rng, state.archive.shape[1], state.archive.shape[0]), -5, 5)
    return x, state.replace(archive=x)

In [30]:
# setup

subject_index = 11
popsize = 50
s_name = "DE"

strategy = Strategies[s_name](popsize=popsize, num_dims=len(parameters["free"]))
es_params = strategy.default_params
state = strategy.initialize(rng, es_params)

single_subject_objective_fn = vmap(
    create_predict_fn(
        model_create_fn,
        subject_presentations[subject_index - 1],
        subject_trials[subject_index - 1],
    )
)

def es_step(state_input, tmp):
    rng, state = state_input
    rng, rng_iter = random.split(rng)

    x, state = lax.cond(
        state.gen_counter == 0,
        lambda _: latin_hypercube_initialize(state, rng_iter),
        lambda _: strategy.ask(rng_iter, state, es_params),
        operand=None,
    )

    x_dict = scale_params(reshape_params(x), parameters["free"])
    fitness = single_subject_objective_fn(x_dict)
    state = strategy.tell(x, fitness, state, es_params)
    return [rng, state], fitness[jnp.argmin(fitness)]

# final_state, scan_out = jax.lax.scan(es_step, [rng, state], [jnp.zeros(1000)])
final_state, scan_out = jax.lax.scan(es_step, [rng, state], None, length=1000)
jnp.min(scan_out)

Array(647.2798, dtype=float32)

I could further compile the initialization step too. The example in the documentation, for example, compiles initialization along with the scan step. 

Furthermore, instead of doing a set number of iterations, I could use lax.while_loop instead and have a stopping condition -- e.g., an absolute change in the loss function value. 

Finally, it may be feasible to use vmap or map to fit multiple datasets at once (not models). init, ask, and tell can all be mapped over a list of models or datasets. And the same can be said for my fitness function. I think? It will of course be tricky to pull this off with varying item counts. I'll get there when I get there.

I need to mind how often compilation happens. If my loss function is insufficiently generic, I would end up recompiling it for every subject. Terrifying. This may require a rewrite of the factory function to avoid conditional function calls.

## Multiple Subjects: Map Function Approach
This approach allows easier code reuse since I don't need a special function for handling multiple subjects. Let's see what it would take to make this work. I will probably want to map over the scan like I current do for `predict_trial`.

That means one function for a single pass. This was `predict_and_simulate_retrieval` in the previous case. It's probably `es_step` in this case. Hopefully I can find a better name. Maybe `increment_fit`? It will return a new state and a fitness value.

Then another function that scans over `es_steps`. This was `predict_trial` in the previous case. It's probably `fit_to_trials` in this case. Hopefully I can find a better name. It will return a final state and a list of states (probably best fitness at each step). I may move this to a while loop instead of scan.

A wrapper function that maps over subjects. This was `predict_trials` mapping over trial vectors in the previous case. Call it `fit_to_subjects`. This one will map over sets of trial vectors. After post-processing the output, it will return a list of fitting parameter configurations as dictionaries.

Main obstacle is that I need to use the same loss function across calls to `fit_to_trials`. This means `create_predict_fn` produces functions that are too specialized. I will ultimately need to address the item_count issue if I want a clean implementation. 

What would that mean? Basically, set array sizes across the model by a max_item_count parameter. But dynamically configure by an item_count parameter, ignoring additional slots when item_count < max_item_count. This would require a rewrite of most of the model functions. Well, maybe not -- perhaps just the accessor functions. No, if the size of arrays returned by accessors depends on item_count, that would raise an error. Ugh. Let's do it; I hate worrying about this.

## Multiple Subjects: Vmap Approach
We will want to vmap the loss_fn twice. Once to handle multiple datasets (sets of subject responses), another to handle multiple N=popsize parameter configurations per dataset. If we can do that across init, ask, and tell, we can fit multiple datasets at once.

In [36]:
es_params

EvoParams(mutate_best_vector=True, num_diff_vectors=1, cross_over_rate=0.9, diff_w=0.8, init_min=-0.1, init_max=0.1, clip_min=-3.4028235e+38, clip_max=3.4028235e+38)

In [35]:
# 1) setup

popsize = 15
s_name = "DE"

strategy = Strategies[s_name](popsize=popsize, num_dims=len(parameters["free"]))
es_params = strategy.default_params

# 2) configure vmap over rng and state
batch_init = jax.vmap(strategy.initialize, in_axes=(0, None)) 
batch_ask = jax.vmap(strategy.ask, in_axes=(0, 0, None))
batch_tell = jax.vmap(strategy.tell, in_axes=(0, 0, 0, None))

batch_latin_hypercube_initialize = jax.vmap(latin_hypercube_initialize, in_axes=(0, 0))

# Vmap-composed objective function
# first by parameter, then by subset of trials
batch_predict = vmap(
    vmap(predict_trials, in_axes=(None, None, None, 0)), in_axes=(None, None, 0, None)
)


# 3) configure scan
def es_step(state_input, tmp):
    rng, state = state_input
    rng, rng_iter = random.split(rng)

    x, state = lax.cond(
        state.gen_counter == 0,
        lambda _: batch_latin_hypercube_initialize(state, rng_iter),
        lambda _: batch_ask(rng_iter, state, es_params),
        operand=None,
    )

    x_dict = scale_params(reshape_params(x), parameters["free"])
    fitness = loss_fn(x_dict)
    state = strategy.tell(x, fitness, state, es_params)
    return [rng, state], fitness[jnp.argmin(fitness)]


# 4) run
rng, rng_iter = random.split(rng)
state = strategy.initialize(rng, es_params)

# final_state, scan_out = jax.lax.scan(es_step, [rng, state], [jnp.zeros(1000)])
final_state, scan_out = jax.lax.scan(es_step, [rng, state], None, length=1000)
jnp.min(scan_out)

Array(642.77075, dtype=float32)